In [13]:
"""
=============================================================================
ABLACAO CONTROLADA DE SCHEDULERS - ResNet50 / Displasia Epitelial Oral (4 classes)
VARIANTE HOLDOUT SIMPLES: SEM cross-validation (sem K-Fold) e SEM ensemble.
Cada configuracao (scheduler, seed) treina UM UNICO modelo usando um split
holdout de 3 vias (treino / validacao / teste) e e' avaliado UMA VEZ no
test set.
=============================================================================

Objetivo: isolar o EFEITO DO SCHEDULER. Igual ao notebook
pipeline_resnet18_no_ensemble.ipynb (mesmas SEEDS, mesmo LR/WEIGHT_DECAY/
BATCH_SIZE, mesma augmentation, mesmos pesos de classe, mesma selecao de
checkpoint), mas SEM StratifiedKFold: em vez de treinar N_FOLDS modelos por
configuracao e reportar media +/- desvio entre eles, este notebook faz um
UNICO split holdout estratificado (treino / validacao / teste) e treina UM
UNICO modelo por (scheduler, seed).

    - mesmas SEEDS  -> mesmo holdout de teste, mesmo holdout de validacao, mesma init
    - mesmo LR, mesmo WEIGHT_DECAY, mesmo BATCH_SIZE
    - mesma augmentation, mesmos pesos de classe, mesma selecao de checkpoint

O UNICO fator que varia e' o scheduler. Assim, qualquer diferenca em F1/QWK/AUC
e' atribuivel ao scheduler, nao a sorte/azar do HPO.

Diferenca vs. pipeline_resnet18_no_ensemble.ipynb:
    - Sem StratifiedKFold / sem loop de folds: 1 modelo por (scheduler, seed)
      em vez de N_FOLDS modelos.
    - A validacao usada para selecionar o checkpoint vem de um holdout
      separado (fatia VAL_SIZE do pool de treino/validacao), nao de folds de CV.
    - Custo computacional ~N_FOLDS vezes menor: nao ha' mais folds para treinar
      nem para medir media/desvio padrao entre eles -- so existe 1 modelo e
      1 avaliacao de teste por configuracao.
    - CSV de log tem colunas mais simples: Val_AUC/Val_F1/Val_QWK/Val_Score e
      Test_F1/Test_AUC/Test_QWK/Test_Accuracy/... sao valores unicos (sem o
      sufixo _Mean/_Std usado no notebook com CV, ja que nao ha' mais fold
      para agregar). A agregacao entre SEEDS no resumo final continua igual.

Correcoes herdadas do notebook original (pipeline_resnet18.ipynb):
    1. Avaliacao de teste roda UMA vez por configuracao, com listas locais
       (sem risco de duplicar a matriz de confusao).
    2. Loga direto QWK, acuracia, erros perigosos (distancia ordinal >=2) e
       subestimacao clinica (predizer classe MENOS grave que a real) -> nao
       precisa recalcular nada depois.
    3. Mapeamento ORDINAL explicito para QWK/erros perigosos (nao depende da
       ordem alfabetica do ImageFolder coincidir com a ordem clinica).
    4. num_workers=0 nos DataLoaders: em notebook Jupyter no Windows, workers
       de multiprocessing (spawn) tentam reimportar __main__ para resolver
       seed_worker, mas __main__ aqui e' o kernel do ipykernel (nao este
       codigo), o que faz os workers crasharem com "DataLoader worker exited
       unexpectedly". Rodando no processo principal evita o crash.

Uso:
    python ablation_schedulers_resnet50_holdout.py
Roda len(SCHEDULERS) x len(SEEDS) experimentos e faz append em um unico CSV.
=============================================================================
"""

import os
import csv
import random
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, f1_score, roc_auc_score, accuracy_score, cohen_kappa_score
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR, ReduceLROnPlateau
from torchvision import datasets, models, transforms
import warnings
import numpy as np

try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    PLOTTING_AVAILABLE = True
except ImportError:
    plt = None
    mticker = None
    PLOTTING_AVAILABLE = False

In [14]:
# =============================================================================
# 1. CONFIGURACAO DA ABLACAO  (estes valores ficam IDENTICOS para os 3 schedulers)
# =============================================================================
SCHEDULERS = ["CosineAnnealingLR", "OneCycleLR", "ReduceLROnPlateau"]
SEEDS      = [42, 17, 7, 12, 123, 555, 996]      # mesmas seeds para os 3 schedulers

# --- Hiperparametros FIXOS (o ponto da ablacao: nao mudar entre schedulers) ---
LR           = 1e-3      # pico de LR (max_lr p/ OneCycle; LR base p/ Cosine/Plateau)
WEIGHT_DECAY = 1e-5
BATCH_SIZE   = 32        # batch 32: o experimento anterior mostrou que 8 e' fragil
EPOCHS       = 50
TEST_SIZE    = 0.2       # fracao do dataset TOTAL reservada para teste
VAL_SIZE     = 0.2       # fracao do pool treino/validacao (o que sobra apos o teste) reservada para validacao
MODEL_NAME   = "ResNet50_Weights.DEFAULT"

# Checkpoint: "auc" preserva o comportamento anterior; "clinical_composite"
# prioriza coerencia ordinal e penaliza erros clinicamente perigosos.
CHECKPOINT_METRIC = "clinical_composite"  # opcoes: "auc", "f1", "qwk", "clinical_composite"
DANGER_RATE_WEIGHT = 1.0
UNDEREST_RATE_WEIGHT = 1.0
SAVE_TELEMETRY = True

# DataLoader workers: 0 = roda no processo principal. Em notebook Jupyter no
# Windows, num_workers>0 com spawn falha ao tentar reimportar __main__ (ver
# nota no topo do arquivo). Se for rodar como script .py standalone (com
# `if __name__ == "__main__":` guard), pode voltar a usar varios workers.
NUM_WORKERS = 0

BASE_DIR   = "Originais"   # ImageFolder, 4 classes, sem split previo
OUTPUT_DIR = f"./ablation_schedulers_resnet50_holdout_ckpt_{CHECKPOINT_METRIC}"
CSV_FILE   = os.path.join(OUTPUT_DIR, "ablation_logs_resnet50_holdout.csv")


os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ordem clinica das classes (rank ordinal). Usada p/ QWK e erros perigosos.
# Se mudar os nomes das classes, ajuste aqui.
ORDINAL_ORDER = ["healthy", "mild", "moderate", "severe"]

# Determinismo: para a ablacao, fixamos a maior reprodutibilidade possivel
# (mesmo split/init/ordem de dados nos 3 schedulers). benchmark=True (do
# notebook original) introduz nao-determinismo de algoritmo de convolucao;
# desligamos aqui para que o scheduler seja a UNICA fonte de variacao.
DETERMINISTIC = True

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

Rodando em: cuda


In [16]:
# =============================================================================
# 2. UTILITARIOS
# =============================================================================
def seed_everything(seed: int):
    """Fixa todas as fontes de aleatoriedade para reprodutibilidade entre schedulers."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

In [17]:
def make_loader_generator(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_model(num_classes: int):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def make_scheduler(name: str, optimizer, steps_per_epoch: int):
    """
    Retorna (scheduler, step_mode).
    step_mode:
      'batch'   -> scheduler.step() apos cada batch          (OneCycleLR)
      'epoch'   -> scheduler.step() ao fim da epoca          (CosineAnnealingLR)
      'plateau' -> scheduler.step(val_loss) ao fim da epoca  (ReduceLROnPlateau)
    """
    if name == "OneCycleLR":
        sch = OneCycleLR(optimizer, max_lr=LR,
                         steps_per_epoch=steps_per_epoch, epochs=EPOCHS)
        return sch, "batch"
    if name == "CosineAnnealingLR":
        sch = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-8)
        return sch, "epoch"
    if name == "ReduceLROnPlateau":
        sch = ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                                patience=5, min_lr=1e-8)
        return sch, "plateau"
    raise ValueError(f"Scheduler desconhecido: {name}")

def ordinal_ranks(class_names):
    """Mapeia o indice de classe do ImageFolder -> rank ordinal clinico (0..3)."""
    missing = [c for c in class_names if c not in ORDINAL_ORDER]
    if missing:
        raise ValueError(
            f"Classes sem rank ordinal: {missing}. Ajuste ORDINAL_ORDER={ORDINAL_ORDER}."
        )
    return np.array([ORDINAL_ORDER.index(c) for c in class_names])

In [18]:
def clinical_metrics(y_true, y_pred, rank_lookup):
    """
    QWK (quadratic), acuracia, erros perigosos e subestimacao clinica.
    Erro perigoso  = distancia ordinal >= 2 (em qualquer direcao).
    Subestimacao   = prever classe MENOS grave que a real, com distancia >= 2
                     (clinicamente o pior caso: chamar de saudavel um doente).
    """
    yt = rank_lookup[np.asarray(y_true)]
    yp = rank_lookup[np.asarray(y_pred)]
    n = len(yt)

    # ?  Mede a coerência entre a previsão e a verdade, penaliza erros que estão mais longes
    qwk = cohen_kappa_score(yt, yp, weights="quadratic")

    acc = accuracy_score(yt, yp)

    # ?  Quantas classes de distância há entre a prevista e a real
    dist = np.abs(yt - yp)
    danger = int((dist >= 2).sum())
    under = int(((yp < yt) & (dist >= 2)).sum())
    return {
        "qwk": qwk,
        "acc": acc,
        "danger_count": danger,
        "danger_rate": danger / n,
        "underest_count": under,
        "underest_rate": under / n,
    }


def checkpoint_selection_score(val_auc, val_f1, y_true, y_pred, rank_lookup):
    """Score usado para escolher o checkpoint salvo no treino holdout."""
    clin = clinical_metrics(y_true, y_pred, rank_lookup)
    metric = CHECKPOINT_METRIC.lower()
    if metric == "auc":
        score = val_auc if not np.isnan(val_auc) else -np.inf
    elif metric == "f1":
        score = val_f1
    elif metric == "qwk":
        score = clin["qwk"]
    elif metric == "clinical_composite":
        score = (clin["qwk"]
                 - DANGER_RATE_WEIGHT * clin["danger_rate"]
                 - UNDEREST_RATE_WEIGHT * clin["underest_rate"])
    else:
        raise ValueError(
            "CHECKPOINT_METRIC deve ser: 'auc', 'f1', 'qwk' ou 'clinical_composite'."
        )
    if np.isnan(score):
        score = -np.inf
    return score, clin

In [19]:
def plot_telemetry(history: dict, scheduler_name: str, seed: int,
                   save_dir: str = OUTPUT_DIR):
    """
    Gera um painel com 3 graficos de linha para auditar o treinamento
    (holdout simples: 1 unico modelo por configuracao).
    history deve conter: 'train_loss', 'val_loss', 'train_acc', 'val_acc', 'lrs'
    """
    if not SAVE_TELEMETRY:
        return None
    if not PLOTTING_AVAILABLE:
        warnings.warn("matplotlib indisponivel; telemetria visual nao sera salva.")
        return None

    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=120)
    fig.suptitle(f"Auditoria de Treinamento - {scheduler_name} | Seed {seed}",
                 fontsize=16, fontweight='bold')

    # --- Painel 1: Loss ---
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], label='Validation Loss', color='#ff7f0e', linewidth=2)
    axes[0].set_title("Evolucao da Funcao de Custo (CrossEntropy)")
    axes[0].set_xlabel("Epocas")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()

    # --- Painel 2: Acuracia ---
    axes[1].plot(epochs, history['train_acc'], label='Train Accuracy', color='#1f77b4', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], label='Validation Accuracy', color='#2ca02c', linewidth=2)
    axes[1].set_title("Evolucao da Acuracia")
    axes[1].set_xlabel("Epocas")
    axes[1].set_ylabel("Acuracia")
    axes[1].set_ylim([0.0, 1.05])
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    # --- Painel 3: Learning Rate ---
    axes[2].plot(epochs, history['lrs'], color='#9467bd', linewidth=2)
    axes[2].set_title(f"Evolucao do Learning Rate ({scheduler_name})")
    axes[2].set_xlabel("Epocas")
    axes[2].set_ylabel("Learning Rate")
    axes[2].grid(True, linestyle='--', alpha=0.6)
    # Forca ticks em intervalos regulares no eixo Y (em vez de depender do
    # auto-scaling, que as vezes nao desenha nenhum rotulo para valores muito
    # pequenos) e formata cada um em notacao cientifica.
    axes[2].yaxis.set_major_locator(mticker.MaxNLocator(nbins=8))
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1e}"))

    plt.tight_layout()

    # Salva o grafico no disco como prova documental
    file_path = os.path.join(save_dir, f"telemetry_{scheduler_name}_seed{seed}.png")
    plt.savefig(file_path)
    print(f" [Grafico] Telemetria salva em: {file_path}")
    plt.close()  # Libera RAM
    return file_path

In [20]:
def run_config(scheduler_name: str, seed: int):
    print("\n" + "#" * 70)
    print(f"# SCHEDULER={scheduler_name}  |  SEED={seed}")
    print("#" * 70)

    seed_everything(seed)
    pin_memory = (device.type == "cuda")
    print(f"Checkpoint metric: {CHECKPOINT_METRIC}")

    data_transforms = {
        "treino": transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(40),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        "teste": transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
    }

    if not os.path.isdir(BASE_DIR):
        raise FileNotFoundError(f"BASE_DIR nao encontrado: {os.path.abspath(BASE_DIR)}")

    full_aug   = datasets.ImageFolder(root=BASE_DIR, transform=data_transforms["treino"])
    full_plain = datasets.ImageFolder(root=BASE_DIR, transform=data_transforms["teste"])
    num_classes = len(full_aug.classes)
    class_names = full_aug.classes
    rank_lookup = ordinal_ranks(class_names)
    all_labels  = np.array(full_aug.targets)

    # --- Quantidade total de imagens e distribuicao por classe ---
    classes_idx, classes_count = np.unique(all_labels, return_counts=True)
    print(f"\n--- Dataset: {len(all_labels)} imagens totais, {num_classes} classes ---")
    for c, cnt in zip(classes_idx, classes_count):
        print(f"    {class_names[c]:<12}: {cnt} imagens")

    # Holdout estratificado em DOIS NIVEIS (sem K-Fold, sem cross-validation):
    #   1) separa test_idx (TEST_SIZE) do resto do dataset;
    #   2) do que sobra (pool treino/validacao), separa val_idx (VAL_SIZE) de train_idx.
    # random_state=seed nos dois splits -> IDENTICO para os 3 schedulers quando
    # a seed e' a mesma. Esse e' o ponto que torna a ablacao "pareada".
    pool_idx, test_idx = train_test_split(
        np.arange(len(all_labels)), test_size=TEST_SIZE,
        stratify=all_labels, random_state=seed,
    )
    pool_labels = all_labels[pool_idx]
    train_pos, val_pos = train_test_split(
        np.arange(len(pool_idx)), test_size=VAL_SIZE,
        stratify=pool_labels, random_state=seed,
    )
    train_idx = pool_idx[train_pos]
    val_idx   = pool_idx[val_pos]

    test_data = Subset(full_plain, test_idx)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                             pin_memory=pin_memory, shuffle=False)

    print(f"--- Split holdout (test_size={TEST_SIZE}, val_size={VAL_SIZE} do pool restante): "
          f"treino={len(train_idx)} | validacao={len(val_idx)} | teste={len(test_idx)} imagens ---")

    train_subset = Subset(full_aug,   train_idx)
    val_subset   = Subset(full_plain, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                              pin_memory=pin_memory, drop_last=True, shuffle=True,
                              worker_init_fn=seed_worker, generator=make_loader_generator(seed))
    val_loader   = DataLoader(val_subset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                              pin_memory=pin_memory, shuffle=False)

    model = build_model(num_classes).to(device)

    train_labels = pool_labels[train_pos]
    cw = compute_class_weight("balanced",
                              classes=np.unique(train_labels),
                              y=train_labels)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(cw, dtype=torch.float32, device=device))
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = len(train_loader)
    scheduler, step_mode = make_scheduler(scheduler_name, optimizer, steps_per_epoch)

    best_score, best_val_auc, best_val_f1 = -np.inf, 0.0, 0.0
    best_val_clin = None
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lrs": []}
    ckpt_path = os.path.join(OUTPUT_DIR, f"{scheduler_name}_seed{seed}_best.pth")

    for epoch in range(EPOCHS):
        # --- treino ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            train_loss += loss.item() * batch_size
            train_correct += (torch.argmax(outputs, 1) == labels).sum().item()
            train_total += batch_size
            if step_mode == "batch":      # OneCycleLR: passo por batch
                scheduler.step()
        avg_train_loss = train_loss / train_total
        train_acc = train_correct / train_total

        # --- validacao ---
        model.eval()
        val_loss, val_total = 0.0, 0
        v_labels, v_preds, v_probs = [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                outputs = model(images)
                batch_size = labels.size(0)
                val_loss += criterion(outputs, labels).item() * batch_size
                val_total += batch_size
                v_labels.extend(labels.cpu().numpy())
                v_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                v_probs.extend(F.softmax(outputs, 1).cpu().numpy())

        avg_val_loss = val_loss / val_total
        val_f1 = f1_score(v_labels, v_preds, average="macro")
        val_acc = accuracy_score(v_labels, v_preds)
        try:
            val_auc = roc_auc_score(v_labels, v_probs,
                                    multi_class="ovr", average="macro")
        except ValueError:
            val_auc = float("nan")
        selection_score, val_clin = checkpoint_selection_score(
            val_auc, val_f1, v_labels, v_preds, rank_lookup
        )

        # --- passo do scheduler por epoca ---
        if step_mode == "epoch":
            scheduler.step()
        elif step_mode == "plateau":
            scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lrs"].append(current_lr)

        print(f"Ep {epoch + 1:02d}/{EPOCHS} | "
              f"LR {current_lr:.2e} | "
              f"TrainLoss {avg_train_loss:.4f} | TrainAcc {train_acc:.4f} | "
              f"ValLoss {avg_val_loss:.4f} | ValAcc {val_acc:.4f} | "
              f"F1 {val_f1:.4f} | AUC {val_auc:.4f} | "
              f"QWK {val_clin['qwk']:.4f} | Score {selection_score:.4f}")

        if selection_score > best_score:
            best_score, best_val_auc, best_val_f1 = selection_score, val_auc, val_f1
            best_val_clin = val_clin
            torch.save(model.state_dict(), ckpt_path)

    if not os.path.isfile(ckpt_path):
        warnings.warn(f"Nenhum checkpoint selecionado em {ckpt_path}; salvando estado final do treino.")
        torch.save(model.state_dict(), ckpt_path)
        best_val_clin = val_clin
    plot_telemetry(history, scheduler_name, seed)
    print(f"--- Melhor checkpoint (holdout) {scheduler_name}/seed{seed}: "
          f"score={best_score:.4f} | AUC={best_val_auc:.4f} | F1={best_val_f1:.4f} | "
          f"QWK={best_val_clin['qwk']:.4f} ---")

    # =========================================================================
    # AVALIACAO NO TEST SET: holdout simples, UM UNICO modelo (sem folds, sem
    # ensemble). O checkpoint com melhor CHECKPOINT_METRIC na validacao e'
    # carregado e avaliado UMA VEZ no test set.
    # =========================================================================
    model = build_model(num_classes).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()

    test_labels, test_preds, test_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            probs = F.softmax(model(images), dim=1)
            test_labels.extend(labels.numpy())
            test_preds.extend(torch.argmax(probs, 1).cpu().numpy())
            test_probs.extend(probs.cpu().numpy())

    test_f1  = f1_score(test_labels, test_preds, average="macro")
    test_auc = roc_auc_score(test_labels, test_probs, multi_class="ovr", average="macro")
    cm       = confusion_matrix(test_labels, test_preds, labels=range(num_classes))
    test_clin = clinical_metrics(test_labels, test_preds, rank_lookup)

    print(f">>> TEST {scheduler_name}/seed{seed} (holdout, 1 modelo): "
          f"F1 {test_f1:.4f} | AUC {test_auc:.4f} | QWK {test_clin['qwk']:.4f} | "
          f"Acc {test_clin['acc']:.4f} | Erros>=2 {test_clin['danger_count']} "
          f"({test_clin['danger_rate']*100:.1f}%)")

    return {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Modelo": MODEL_NAME,
        "Scheduler": scheduler_name,
        "Seed": seed,
        "Epochs": EPOCHS, "Batch_Size": BATCH_SIZE,
        "LR": LR, "Weight_Decay": WEIGHT_DECAY, "Optimizer": "AdamW",
        "Checkpoint_Metric": CHECKPOINT_METRIC,
        "Classes": str(class_names), "Test_Size": TEST_SIZE, "Val_Size": VAL_SIZE,
        "Val_AUC": f"{best_val_auc:.4f}", "Val_F1": f"{best_val_f1:.4f}",
        "Val_QWK": f"{best_val_clin['qwk']:.4f}", "Val_Score": f"{best_score:.4f}",
        "Val_Danger_Rate": f"{best_val_clin['danger_rate']:.4f}",
        "Val_Underest_Rate": f"{best_val_clin['underest_rate']:.4f}",
        "Test_F1": f"{test_f1:.4f}", "Test_AUC": f"{test_auc:.4f}",
        "Test_QWK": f"{test_clin['qwk']:.4f}", "Test_Accuracy": f"{test_clin['acc']:.4f}",
        "Test_Danger_Rate": f"{test_clin['danger_rate']:.4f}",
        "Test_Underest_Rate": f"{test_clin['underest_rate']:.4f}",
        "Test_Confusion_Matrix": str(cm.tolist()),
    }

In [21]:
# =============================================================================
# 4. LOOP DA ABLACAO  +  LOGGING
# =============================================================================
FIELDNAMES = [
    "Timestamp", "Modelo", "Scheduler", "Seed", "Epochs", "Batch_Size",
    "LR", "Weight_Decay", "Optimizer", "Checkpoint_Metric", "Classes",
    "Test_Size", "Val_Size",
    "Val_AUC", "Val_F1", "Val_QWK", "Val_Score",
    "Val_Danger_Rate", "Val_Underest_Rate",
    "Test_F1", "Test_AUC", "Test_QWK", "Test_Accuracy",
    "Test_Danger_Rate", "Test_Underest_Rate",
    "Test_Confusion_Matrix",
]


def log_row(row: dict):
    exists = os.path.isfile(CSV_FILE)
    if exists:
        with open(CSV_FILE, newline="", encoding="utf-8") as f:
            header = next(csv.reader(f), [])
        if header != FIELDNAMES:
            raise ValueError(
                f"CSV existente tem colunas incompativeis com o pipeline atual: {CSV_FILE}"
            )
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def get_completed_configs(csv_path: str = CSV_FILE) -> set:
    """
    Le o CSV de resultados (se existir) e retorna o conjunto de (Scheduler, Seed)
    ja concluidos, para que main() possa pular essas combinacoes ao retomar uma
    ablacao interrompida no meio (crash, falta de energia, etc.).
    """
    if not os.path.isfile(csv_path):
        return set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {(row["Scheduler"], int(row["Seed"])) for row in reader}


# =============================================================================
# 5. RESUMO ESTATISTICO DA ABLACAO (le o CSV; nao re-treina)
# =============================================================================
import ast


def _expand_cm(cm):
    """Matriz de confusao -> (y_true, y_pred) repetindo cada celula."""
    yt, yp = [], []
    for i, row in enumerate(cm):
        for j, c in enumerate(row):
            yt += [i] * int(c)
            yp += [j] * int(c)
    return np.array(yt), np.array(yp)

In [22]:
def summarize_ablation(csv_path: str = CSV_FILE):
    import pandas as pd
    try:
        from scipy import stats
        have_scipy = True
    except ImportError:
        have_scipy = False

    if not os.path.isfile(csv_path):
        print(f"[aviso] CSV nao encontrado: {csv_path}")
        print("        Rode main(run_training=True) para gerar os resultados desta configuracao.")
        return

    df = pd.read_csv(csv_path)
    df = _ensure_metric_cols(df)
    for col in ["Test_QWK", "Test_F1", "Test_AUC", "Test_Danger_Rate", "Test_Accuracy"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    metrics = [("Test_QWK", "QWK", "max"),
               ("Test_F1", "F1", "max"),
               ("Test_AUC", "AUC", "max"),
               ("Test_Accuracy", "Acc", "max"),
               ("Test_Danger_Rate", "Erro>=2", "min")]

    print("\n" + "=" * 78)
    print(" RESUMO POR SCHEDULER  (media +/- desvio entre seeds; holdout simples, 1 modelo/config)")
    print("=" * 78)
    order = [s for s in SCHEDULERS if s in df["Scheduler"].unique()]
    extra = [s for s in df["Scheduler"].unique() if s not in order]
    order += extra
    head = f"{'Scheduler':<20}" + "".join(f"{lbl:<16}" for _, lbl, _ in metrics) + "n"
    print(head)
    print("-" * len(head))
    for sch in order:
        g = df[df["Scheduler"] == sch]
        cells = ""
        for col, _, _ in metrics:
            cells += f"{g[col].mean():.4f}+/-{g[col].std(ddof=1):.3f}  "
        print(f"{sch:<20}{cells}{len(g)}")

    # Vencedor por metrica (melhor media; '*' marca o melhor)
    print("\nMelhor media por metrica:")
    for col, lbl, direction in metrics:
        means = df.groupby("Scheduler")[col].mean()
        best = means.idxmin() if direction == "min" else means.idxmax()
        print(f"  {lbl:<10} -> {best}  ({means[best]:.4f})")

    if not have_scipy:
        print("\n[aviso] scipy nao instalado: pulei os testes de significancia.")
        print("        instale com: pip install scipy")
        return

    # --- Tabela pareada: linhas=seed, colunas=scheduler ---
    print("\n" + "=" * 78)
    print(" TESTES DE SIGNIFICANCIA PAREADOS (pareados por seed)")
    print("=" * 78)
    n_seeds = df["Seed"].nunique()
    print(f"n_seeds={n_seeds}.", end=" ")
    if n_seeds < 6:
        print("ATENCAO: com n<6, o Wilcoxon nao alcanca p<0.05 mesmo no caso\n"
              "         extremo (p minimo bilateral = 0.0625). Trate como exploratorio;\n"
              "         para conclusao formal, aumente o numero de seeds.")
    else:
        print("")

    for col, lbl, direction in metrics:
        piv = df.pivot_table(index="Seed", columns="Scheduler",
                             values=col, aggfunc="mean")
        schs = [s for s in order if s in piv.columns]
        sub = piv[schs].dropna(axis=0, how="any")  # so seeds presentes em todos
        if len(sub) < 2 or len(schs) < 2:
            print(f"\n[{lbl}] dados insuficientes para teste pareado.")
            continue

        print(f"\n[{lbl}]  (seeds pareadas usadas: {len(sub)})")
        # Omnibus Friedman (>=3 schedulers)
        if len(schs) >= 3:
            try:
                chi2, p = stats.friedmanchisquare(*[sub[s].values for s in schs])
                print(f"  Friedman (omnibus): chi2={chi2:.3f}  p={p:.4f}")
            except Exception as e:
                print(f"  Friedman: nao aplicavel ({e})")
        # Post-hoc pareado par a par
        better = max if direction == "max" else min
        for a in range(len(schs)):
            for b in range(a + 1, len(schs)):
                xa, xb = sub[schs[a]].values, sub[schs[b]].values
                diff = xa - xb
                try:
                    w_p = stats.wilcoxon(xa, xb, zero_method="wilcox").pvalue
                except ValueError:
                    w_p = float("nan")  # ex.: todas as diferencas iguais a zero
                t_p = stats.ttest_rel(xa, xb).pvalue
                win = schs[a] if better(xa.mean(), xb.mean()) == xa.mean() else schs[b]
                print(f"    {schs[a]:<18} vs {schs[b]:<18} | "
                      f"d_medio={diff.mean():+.4f} | Wilcoxon p={w_p:.4f} | "
                      f"t-pareado p={t_p:.4f} | tende a favor de: {win}")

    print("\n[nota] QWK/F1/AUC/Acc: maior e' melhor. Erro>=2: menor e' melhor.")
    print("=" * 78)

In [23]:

def _ensure_metric_cols(df):
    """
    Garante colunas Test_QWK / Test_Danger_Rate / Test_Accuracy.
    Se faltarem (ex.: CSVs antigos), recalcula a partir de
    Test_Confusion_Matrix usando a ordem ORDINAL clinica (nao a ordem
    alfabetica do ImageFolder).
    """
    need = {"Test_QWK", "Test_Danger_Rate", "Test_Accuracy"} - set(df.columns)
    if not need:
        return df
    qwks, drates, accs = [], [], []
    for _, r in df.iterrows():
        try:
            class_names = ast.literal_eval(r["Classes"])
        except Exception:
            class_names = ORDINAL_ORDER
        rank_lookup = ordinal_ranks(class_names)
        cm = np.array(ast.literal_eval(r["Test_Confusion_Matrix"]))
        yt, yp = _expand_cm(cm.tolist())
        m = clinical_metrics(yt, yp, rank_lookup)
        qwks.append(m["qwk"]); drates.append(m["danger_rate"]); accs.append(m["acc"])
    if "Test_QWK" not in df.columns:          df["Test_QWK"] = qwks
    if "Test_Danger_Rate" not in df.columns:  df["Test_Danger_Rate"] = drates
    if "Test_Accuracy" not in df.columns:     df["Test_Accuracy"] = accs
    return df

In [24]:

def main(run_training: bool = True):
    if run_training:
        # Resume: pula combinacoes (scheduler, seed) que ja estao no CSV, para
        # que uma ablacao interrompida no meio (crash, falta de energia, etc.)
        # possa ser retomada sem re-treinar o que ja foi concluido.
        completed = get_completed_configs(CSV_FILE)
        if completed:
            print(f"[resume] {len(completed)} combinacoes ja concluidas em {CSV_FILE}; serao puladas.")

        # Loop por scheduler (externo) e por seed (interno). Para uma dada seed,
        # os 3 schedulers veem o MESMO split de teste e o MESMO split de validacao.
        for scheduler_name in SCHEDULERS:
            for seed in SEEDS:
                if (scheduler_name, seed) in completed:
                    print(f"[SKIP] {scheduler_name}/seed{seed} ja esta em {CSV_FILE}")
                    continue
                row = run_config(scheduler_name, seed)
                log_row(row)
                completed.add((scheduler_name, seed))
                print(f"[LOG] {scheduler_name}/seed{seed} salvo em {CSV_FILE}")
        print("\n" + "=" * 70)
        print(f"Ablacao concluida: {len(SCHEDULERS) * len(SEEDS)} experimentos.")
        print(f"Resultados em: {CSV_FILE}")
        print("=" * 70)

    # Resumo estatistico ao final (sempre roda; le o CSV gerado)
    summarize_ablation(CSV_FILE)


if __name__ == "__main__":
    import sys
    # --summary [caminho.csv]  -> so resume um CSV existente (nao re-treina).
    # Funciona tambem nos CSVs antigos (recalcula QWK/erros pela matriz).
    if len(sys.argv) > 1 and sys.argv[1] == "--summary":
        path = sys.argv[2] if len(sys.argv) > 2 else CSV_FILE
        summarize_ablation(path)
        sys.exit(0)
    main()

[resume] 21 combinacoes ja concluidas em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.csv; serao puladas.
[SKIP] CosineAnnealingLR/seed42 ja esta em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.csv
[SKIP] CosineAnnealingLR/seed17 ja esta em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.csv
[SKIP] CosineAnnealingLR/seed7 ja esta em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.csv
[SKIP] CosineAnnealingLR/seed12 ja esta em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.csv
[SKIP] CosineAnnealingLR/seed123 ja esta em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.csv
[SKIP] CosineAnnealingLR/seed555 ja esta em ./ablation_schedulers_resnet50_holdout_ckpt_clinical_composite\ablation_logs_resnet50_holdout.cs